# 💻👔💼 <span style="color: white; background-color: Indigo"><b> Extração da Base de Cargos </b></span></p>

🧩 <span style="color: Orchid"><b> 1- Acesso Automatizado ao Datamace </b></span></p>
- Carrega credenciais via variáveis de ambiente (.env)
- Verifica se a interface HTML5 já está aberta
- Caso não esteja, abre o navegador via Selenium, realiza o login no portal Cloud e aguarda o carregamento
- Utiliza PyAutoGUI para realizar o login interno no sistema
- Isso garante que a extração ocorra de forma fluida e sem intervenção manual na autenticação

📥 <span style="color: Orchid"><b> 2- Navegação e Extração do Relatório </b></span></p>
- Acessa o módulo CS (Cargos e Salários)
- Navega até a Tabela de Cargos
- Configura a saída para Excel e dispara o download
- Exibe um prompt de confirmação: "Foi realizado o download da base de CARGOS?" para garantir a integridade antes de avançar.

🧹 <span style="color: Orchid"><b> 3- Carga, Limpeza e Padronização da Base </b></span></p>
- O script busca o arquivo baixado (padrão GPS*.XLSX) e executa o tratamento:
    - Renomeação e Arquivamento
- Renomeia o arquivo com a data atual e, após o processamento, move o arquivo original para a pasta ARQUIVOS MOVIDOS.
    - Mapeamento de Colunas
- Transforma as colunas do Datamace para o padrão interno de People Analytics
- Gera o arquivo oficial estruturado: 📘 CARGOS_DATAMACE.xlsx (com formatação TableStyleLight13)

🕵️‍♂️ <span style="color: Orchid"><b> 4- Módulo de Validação e Auditoria (Regras de Negócio) </b></span></p>
- Esta é a etapa mais analítica do projeto
- O script cruza a base nova com a planilha de controle interno (CARGOS E SALÁRIOS.xlsx) e a base de COLABORADORES.xlsx, aplicando 5 regras de validação estritas:
    - Cargos não cadastrados: Identifica cargos novos (ID >= 834) que estão no Datamace, mas faltam no controle do GP.
    - Cargos antigos ativos: Verifica se cargos legados (anteriores ao reenquadramento, ID < 834) ainda possuem status "Ativo".
    - Descrição completa alterada: Aponta divergências na nomenclatura completa do cargo entre o sistema e a tabela do RH.
    - Descrição resumida alterada: Aponta divergências na nomenclatura abreviada.
    - Divergências salariais: Cruza o salário atual dos colaboradores ativos com o salário estipulado na tabela de cargos, listando qualquer diferença (centavos ou reajustes perdidos).

🌐 <span style="color: Orchid"><b> 5- Geração do Relatório HTML de Auditoria </b></span></p>
- Com os resultados da validação, o script gera automaticamente um relatório:
    - CARGOS E SALÁRIOS - DD-MM-YYYY.html
- Construído com HTML e CSS injetados via Python
- Tabelas estilizadas e coloridas (padrão corporativo)
- Destaca a quantidade de erros em cada uma das 5 regras de validação
- Pronto para ser aberto no navegador, revisado pela gestão ou enviado por e-mail

🔄 <span style="color: Orchid"><b> 6- Atualização Automática do Power BI </b></span></p>
- Abre o arquivo CARGOS E SALÁRIOS.pbix
- Clica em Atualizar e aguarda o processamento
- Clica em Publicar no workspace GESTÃO DE PESSOAS
- Confirma a substituição do relatório antigo
- Fecha o Power BI
- O dashboard corporativo passa a refletir a estrutura de cargos mais recente

🗃️ <span style="color: Orchid"><b> 7- Exportação para Banco de Dados e Logging </b></span></p>
- Converte a base final para tb_cargos.csv (pronto para SQL/DW)
- Registra todas as etapas (0, 1, 2 e possíveis erros) no arquivo PROCESSOS.xlsx
- Calcula e exibe o tempo total de execução no console

# Importando as Bibliotecas

In [9]:
import logging
import os
import pygetwindow as gw
import pandas as pd
import pyautogui
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from IPython.display import HTML, display
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [10]:
# Constantes de configuração

PATH_REGISTROS_PROCESSOS = Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx')
DIRETORIO_BUSCAR = Path(r'C:\Users\rodrigo.bernandes\Downloads')
DIRETORIO_MOVER = Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS')
CAMINHO_ARQUIVO_FINAL = Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS_DATAMACE.xlsx')
CAMINHO_CARGOS_SALARIOS = Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS E SALÁRIOS.xlsx')
CAMINHO_COLABORADORES = Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx')
ENV_PATH = Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env')
ID_CARGA = 6  # ID para carga
ID_VALIDACAO = 19  # ID para validação

# Registra Etapa do Processo

In [11]:
def registrar_etapa(id_processo, etapa, wb_p=None, ws_p=None):
    """Registra uma etapa no controle de processos."""
    linha = [id_processo, datetime.today(), etapa]
    if ws_p is None:
        wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
        ws_p = wb_p['REGISTROS']
    ws_p.append(linha)
    wb_p.save(PATH_REGISTROS_PROCESSOS)
    logger.info(f"Etapa {etapa} registrada com sucesso.")
    return wb_p, ws_p

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [12]:
load_dotenv(dotenv_path=ENV_PATH)

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o CS

In [13]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o CS
time.sleep(10) # Tempo maior para aguardar carregar o CS
pyautogui.click(x=273, y=195) # Clicando no módulo CS
time.sleep(3)
pyautogui.click(x=1079, y=235) # Fecha a janela de Tarefas Anuais
time.sleep(3)
pyautogui.click(x=449, y=297) # Cargos e Salários
time.sleep(2)
pyautogui.click(x=578, y=322) # Tabela Cargos/Funções
time.sleep(2)
pyautogui.click(x=710, y=322) # Tabela de Cargos
time.sleep(2)
pyautogui.click(x=907, y=724) # Imprimir
time.sleep(2)
pyautogui.click(x=762, y=603) # Saída
time.sleep(2)
pyautogui.click(x=778, y=686) # Excel
pyautogui.press("tab")
time.sleep(2)
pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da Base CARGOS

In [14]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base de CARGOS?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-26 10:10:23,048 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-26 10:10:23,049 - INFO - Iniciando a leitura e processamento dos dados...


# Processo de Carga e Geração do Arquivo em HTML

In [15]:
def processo_carga():
    """Processo de carga de cargos do Datamace."""
    logger.info("Iniciando processo de carga.")
    
    if not PATH_REGISTROS_PROCESSOS.exists():
        raise FileNotFoundError("Arquivo de registros de processos não encontrado.")
    
    wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
    ws_p = wb_p['REGISTROS']
    
    # Etapa 0: Início
    wb_p, ws_p = registrar_etapa(ID_CARGA, 0, wb_p, ws_p)
    
    # Verificar diretórios
    if not DIRETORIO_BUSCAR.exists():
        raise FileNotFoundError(f"Diretório de busca não encontrado: {DIRETORIO_BUSCAR}")
    if not DIRETORIO_MOVER.exists():
        DIRETORIO_MOVER.mkdir(parents=True, exist_ok=True)
    
    # Listar e filtrar arquivos GPS
    arquivos_gps = [f for f in DIRETORIO_BUSCAR.glob("*.XLSX") if f.name.startswith('GPS')]
    if not arquivos_gps:
        logger.error("Nenhum arquivo GPS encontrado no diretório.")
        wb_p, ws_p = registrar_etapa(ID_CARGA, 19, wb_p, ws_p)  # Etapa de erro
        raise FileNotFoundError("Nenhum arquivo GPS encontrado no diretório.")
    
    logger.info(f"Encontrados {len(arquivos_gps)} arquivos GPS.")
    
    # Etapa 1: Preparação
    wb_p, ws_p = registrar_etapa(ID_CARGA, 1, wb_p, ws_p)
    
    data_hoje = datetime.today().strftime('%Y-%m-%d')
    for arquivo in arquivos_gps:
        logger.info(f"Processando arquivo: {arquivo.name}")
        try:
            # Renomear para incluir data
            nome_arquivo_com_data = f"GPS_{data_hoje}.XLSX"
            arquivo_com_data = DIRETORIO_BUSCAR / nome_arquivo_com_data
            arquivo.rename(arquivo_com_data)
            
            # Carregar e tratar dados
            cargos = pd.read_excel(arquivo_com_data)
            
            # Verificar colunas necessárias (com acentos)
            colunas_esperadas = ['NÚMERO', 'CARGO', 'CG_DES_COM', 'CBO', 'STATUS', 'LEI APREND']
            colunas_faltantes = [c for c in colunas_esperadas if c not in cargos.columns]
            if colunas_faltantes:
                raise ValueError(f"Colunas faltantes no arquivo: {colunas_faltantes}")
            
            # Renomeando colunas
            cargos_tratado = cargos[['NÚMERO', 'CARGO', 'CG_DES_COM', 'CBO', 'STATUS', 'LEI APREND']].copy()
            cargos_tratado.rename(columns={
                'NÚMERO': 'id_Cargo',
                'CARGO': 'cargo_abreviado',
                'CG_DES_COM': 'cargo_completo',
                'CBO': 'cbo',
                'STATUS': 'status',
                'LEI APREND': 'lei_aprendiz'
            }, inplace=True)
            
            # Criar Excel com tabela formatada
            wb = Workbook()
            ws = wb.active
            ws.title = "CARGOS"
            
            for r in dataframe_to_rows(cargos_tratado, index=False, header=True):
                ws.append(r)
            
            # Adicionar tabela
            tabela_cargos = Table(displayName="CARGOSS", ref=ws.dimensions)
            estilo_tabela = TableStyleInfo(
                name="TableStyleLight13",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=True
            )
            tabela_cargos.tableStyleInfo = estilo_tabela
            ws.add_table(tabela_cargos)
            
            wb.save(CAMINHO_ARQUIVO_FINAL)
            logger.info(f"Arquivo salvo: {CAMINHO_ARQUIVO_FINAL}")
            
            # Mover arquivo processado
            destino = DIRETORIO_MOVER / nome_arquivo_com_data
            shutil.move(arquivo_com_data, destino)
            logger.info(f"Arquivo movido para: {destino}")
            
        except Exception as e:
            logger.error(f"Erro ao processar {arquivo.name}: {str(e)}")
            # Reverter renomeação se necessário
            if 'arquivo_com_data' in locals() and arquivo_com_data.exists():
                arquivo_com_data.rename(arquivo)
    
    # Etapa 2: Finalização
    wb_p, ws_p = registrar_etapa(ID_CARGA, 2, wb_p, ws_p)
    logger.info("Processo de carga finalizado com sucesso.")

def limpar_coluna_numerica(df, coluna):
    """Limpa e converte coluna para inteiro, removendo não-numéricos."""
    logger.info(f"Limpando coluna '{coluna}'.")
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
    linhas_originais = len(df)
    df = df.dropna(subset=[coluna])
    df[coluna] = df[coluna].astype(int)
    linhas_removidas = linhas_originais - len(df)
    logger.info(f"Removidas {linhas_removidas} linhas não-numéricas em '{coluna}'.")
    if linhas_removidas > 0:
        logger.warning(f"AVISO: {linhas_removidas} linhas removidas de '{coluna}'.")
    return df

def processo_validacao():
    """Processo de validação de cargos e salários."""
    logger.info("Iniciando processo de validação.")
    
    if not PATH_REGISTROS_PROCESSOS.exists():
        raise FileNotFoundError("Arquivo de registros de processos não encontrado.")
    
    wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
    ws_p = wb_p['REGISTROS']
    
    # Etapa 0: Início
    wb_p, ws_p = registrar_etapa(ID_VALIDACAO, 0, wb_p, ws_p)
    
    hoje = datetime.today().date()
    data_br = hoje.strftime('%d/%m/%Y')
    caminho_arquivo_html = Path(r'X:\Gestão de Pessoas\Analytics\11 - Validações\11.1 - Cargos e Salários\2026') / f"CARGOS E SALÁRIOS - {data_br.replace('/', '-')}.html"
    
    # Verificar arquivos de entrada
    arquivos_necessarios = [CAMINHO_ARQUIVO_FINAL, CAMINHO_CARGOS_SALARIOS, CAMINHO_COLABORADORES]
    for arq in arquivos_necessarios:
        if not arq.exists():
            raise FileNotFoundError(f"Arquivo necessário não encontrado: {arq}")
    
    # Carregar bases
    cargos_datamace = pd.read_excel(CAMINHO_ARQUIVO_FINAL)
    cargos_salarios = pd.read_excel(CAMINHO_CARGOS_SALARIOS, usecols='B:L', skiprows=3)
    colaboradores = pd.read_excel(CAMINHO_COLABORADORES)
    
    # Limpeza
    cargos_datamace = limpar_coluna_numerica(cargos_datamace, 'id_Cargo')
    cargos_salarios = limpar_coluna_numerica(cargos_salarios, 'ID Datamace')
    
    logger.info("Bases carregadas e limpas.")
    
    # 1. Cargos não cadastrados no controle GP (após reenquadramento)
    merged_1 = pd.merge(cargos_datamace, cargos_salarios[['ID Datamace']], left_on='id_Cargo', right_on='ID Datamace', how='left')
    df_nao_cad_controle = merged_1[(merged_1['ID Datamace'].isna()) & (merged_1['id_Cargo'] >= 834)]
    df_nao_cad_controle = df_nao_cad_controle[['id_Cargo', 'cargo_completo']].rename(columns={
        'id_Cargo': 'ID Datamace',
        'cargo_completo': 'Cargo Completo (Datamace)'
    }).sort_values('ID Datamace')
    
    nao_cad_controle = len(df_nao_cad_controle)
    logger.info(f"Validação 1 executada: {nao_cad_controle} cargos não cadastrados no controle GP.")
    
    # 2. Cargos antigos ativos
    df_antigos_ativos = cargos_datamace[(cargos_datamace['id_Cargo'] < 834) & (cargos_datamace['status'] == 'Ativo')]
    df_antigos_ativos = df_antigos_ativos[['id_Cargo', 'cargo_completo']].rename(columns={
        'id_Cargo': 'ID Datamace',
        'cargo_completo': 'Cargo Completo'
    }).sort_values('ID Datamace')
    
    antigos_ativos = len(df_antigos_ativos)
    logger.info(f"Validação 2 executada: {antigos_ativos} cargos antigos ativos.")
    
    # 3. Descrição completa alterada
    merged_3 = pd.merge(cargos_datamace, cargos_salarios, left_on='id_Cargo', right_on='ID Datamace', how='inner')
    df_descricao_completa = merged_3[(merged_3['id_Cargo'] >= 834) & (merged_3['cargo_completo'] != merged_3['Descrição Completa'])]
    df_descricao_completa = df_descricao_completa[['id_Cargo', 'cargo_completo', 'Descrição Completa']].rename(columns={
        'id_Cargo': 'ID Datamace',
        'cargo_completo': 'Descrição Completa (Datamace)',
        'Descrição Completa': 'Descrição Completa (Tabela GP)'
    }).sort_values('ID Datamace')
    
    descricao_completa = len(df_descricao_completa)
    logger.info(f"Validação 3 executada: {descricao_completa} cargos com descrição completa alterada.")
    
    # 4. Descrição resumida alterada
    merged_4 = pd.merge(cargos_datamace, cargos_salarios, left_on='id_Cargo', right_on='ID Datamace', how='inner')
    df_descricao_resumida = merged_4[(merged_4['id_Cargo'] >= 834) & (merged_4['cargo_abreviado'] != merged_4['Descrição Resumida'])]
    df_descricao_resumida = df_descricao_resumida[['id_Cargo', 'cargo_abreviado', 'Descrição Resumida']].rename(columns={
        'id_Cargo': 'ID Datamace',
        'cargo_abreviado': 'Descrição Resumida (Datamace)',
        'Descrição Resumida': 'Descrição Resumida (Tabela GP)'
    }).sort_values('ID Datamace')
    
    descricao_resumida = len(df_descricao_resumida)
    logger.info(f"Validação 4 executada: {descricao_resumida} cargos com descrição resumida alterada.")
    
    # 5. Divergências salariais
    merged_5 = pd.merge(colaboradores, cargos_salarios, left_on='cargo_completo', right_on='Descrição Completa', how='left')
    df_salario_dif = merged_5[(merged_5['salario_atual'] != merged_5['Salário']) & (merged_5['descricao_rescisao'] == 'ATIVO')]
    df_salario_dif = df_salario_dif[['registro', 'nome', 'cargo_completo', 'centro_custo', 'salario_atual', 'Salário', 'hora_base']].rename(columns={
        'registro': 'Matrícula',
        'nome': 'Nome do colaborador',
        'cargo_completo': 'Cargo',
        'centro_custo': 'Centro de custo',
        'salario_atual': 'Salário praticado',
        'Salário': 'Salário da tabela',
        'hora_base': 'Carga horária'
    }).sort_values('Cargo')
    
    salario_dif = len(df_salario_dif)
    logger.info(f"Validação 5 executada: {salario_dif} colaboradores com salário divergente.")
    
    # CSS para tabelas
    css = """
    <style>
        .minha-tabela { border-collapse: collapse; width: 80%; margin: auto; }
        .minha-tabela th, .minha-tabela td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        .minha-tabela th { background-color: #377AB4; color: white; }
        .minha-tabela tr:nth-child(even) { background-color: #f2f2f2; }
        .minha-tabela tr:hover { background-color: #ddd; }
    </style>
    """
    
    # Gerar HTML
    html_titulo = f"""
    <div style="background-color: #23A638; padding: 1px; border: 3px solid #23A638;">
        <h1 style="color: #FFFFFF; font-size: 28px; font-family: 'Verdana'; font-weight: bold;"> VALIDAÇÃO DE CARGOS E SALÁRIOS - {data_br}</h1>
    </div>
    """
    
    # 1. Cargos não cadastrados
    html_nao_cad = df_nao_cad_controle.to_html(index=False, classes='minha-tabela', escape=False) if not df_nao_cad_controle.empty else "<p>Nenhum cargo não cadastrado encontrado.</p>"
    html_val_1 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos cadastrados após reenquadramento que não estão no controle GP: {nao_cad_controle}</p>
    </div>
    {html_nao_cad}
    """
    
    # 2. Cargos antigos ativos
    html_antigos = df_antigos_ativos.to_html(index=False, classes='minha-tabela', escape=False) if not df_antigos_ativos.empty else "<p>Nenhum cargo antigo ativo encontrado.</p>"
    html_val_2 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos antigos (anteriores ao reenquadramento) que ainda estão ativos: {antigos_ativos}</p>
    </div>
    {html_antigos}
    """
    
    # 3. Descrição completa alterada
    html_desc_completa = df_descricao_completa.to_html(index=False, classes='minha-tabela', escape=False) if not df_descricao_completa.empty else "<p>Nenhuma alteração na descrição completa encontrada.</p>"
    html_val_3 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição completa alterada: {descricao_completa}</p>
    </div>
    {html_desc_completa}
    """
    
    # 4. Descrição resumida alterada
    html_desc_resumida = df_descricao_resumida.to_html(index=False, classes='minha-tabela', escape=False) if not df_descricao_resumida.empty else "<p>Nenhuma alteração na descrição resumida encontrada.</p>"
    html_val_4 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição resumida alterada: {descricao_resumida}</p>
    </div>
    {html_desc_resumida}
    """
    
    # 5. Divergências salariais
    html_salario_dif = df_salario_dif.to_html(index=False, classes='minha-tabela', escape=False) if not df_salario_dif.empty else "<p>Nenhuma divergência salarial encontrada.</p>"
    html_val_5 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de colaboradores com o salário divergente da tabela: {salario_dif}</p>
    </div>
    {html_salario_dif}
    """
    
    html_resumo = css + '<br>' + html_titulo + '<br>' + html_val_1 + '<br>' + html_val_2 + '<br>' + html_val_3 + '<br>' + html_val_4 + '<br>' + html_val_5 + '<br>'
    
    # Salvar HTML
    with open(caminho_arquivo_html, 'w', encoding='utf-8') as file:
        file.write(html_resumo)
    logger.info(f"Relatório HTML salvo em: {caminho_arquivo_html}")
    
    # Etapa 2: Finalização
    wb_p, ws_p = registrar_etapa(ID_VALIDACAO, 2, wb_p, ws_p)
    logger.info("Processo de validação finalizado com sucesso.")

def main():
    """Função principal: Executa carga seguida de validação."""
    logger.info("Iniciando fluxo completo: Carga + Validação.")
    try:
        processo_carga()
        processo_validacao()
        logger.info("Fluxo completo executado com sucesso.")
    except Exception as e:
        logger.error(f"Erro no fluxo principal: {str(e)}")
        # Registrar erro final no processo
        wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
        ws_p = wb_p['REGISTROS']
        registrar_etapa(ID_CARGA if 'carga' in str(e).lower() else ID_VALIDACAO, 19, wb_p, ws_p)
        raise

if __name__ == "__main__":
    main()

2026-06-26 10:10:23,104 - INFO - Iniciando fluxo completo: Carga + Validação.
2026-06-26 10:10:23,107 - INFO - Iniciando processo de carga.
2026-06-26 10:10:23,534 - INFO - Etapa 0 registrada com sucesso.
2026-06-26 10:10:23,540 - INFO - Encontrados 1 arquivos GPS.
2026-06-26 10:10:23,731 - INFO - Etapa 1 registrada com sucesso.
2026-06-26 10:10:23,731 - INFO - Processando arquivo: GPS017-SC-10100808.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-26 10:10:24,580 - INFO - Arquivo salvo: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS_DATAMACE.xlsx
2026-06-26 10:10:24,653 - INFO - Arquivo movido para: X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS\GPS_2026-06-26.XLSX
2026-06-26 10:10:24,855 - INFO - Etapa 2 registrada com suces

# Atualização do Power BI - CARGOS E SALÁRIOS

In [16]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\CARGOS E SALÁRIOS.pbix"
os.startfile(path_pbi) # Abre o arquivo
time.sleep(40)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(3)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI Atualizado

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [17]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS_DATAMACE.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_cargos.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [18]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:03:28.431986

----------------------------------------------------------------------------------------------------
